# Human Validation Study — Survey Analysis

**Thesis Part B**: Validating the hybrid NLP Greenwashing metric against human judgement.

This notebook analyses the Prolific/SoSci survey data where domain experts rated
sustainability report paragraphs on **specificity** and **greenwashing perception**
(5-point Likert scales). It computes inter-rater reliability (Krippendorff's α),
correlates human ratings with the automated pipeline scores (VUI, SPI), and runs
the SFDR sub-sample comparison.

**To update with new data**: change `CSV_PATH` in the next cell and re-run all.

In [ ]:
# ── Cell 0: Setup & Config ───────────────────────────────────────────────────
import warnings, json, os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import krippendorff
import statsmodels.api as sm

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths (edit CSV_PATH when new data arrives) ─────────────────────────────
REPO = Path(os.path.abspath("")).parent  # assumes notebook lives in scripts/
CSV_PATH      = REPO / "data" / "sosci_export.csv"
PROLIFIC_PATH = REPO / "data" / "prolific_demographics.csv"
KEY_PATH      = REPO / "archive_survey_drafts" / "prolific_survey_key.csv"
BATCH_DIR = REPO / "outputs" / "batch"

# ── Quality-filter thresholds (configurable) ────────────────────────────────
SPEED_THRESHOLD_SECS = 180   # participants faster than this are flagged
GOLD_ACCURACY_THRESHOLD = 0.0   # disabled — gold items too ambiguous for reliable filtering

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.1)
PAL = sns.color_palette("colorblind")
BAND_COLORS = {
    "very_low": "#27ae60", "low": "#2ecc71",
    "medium": "#f39c12", "high": "#e74c3c", "very_high": "#8e44ad",
}

# ── DM label maps ───────────────────────────────────────────────────────────
DM01_LABELS = {1: "High School", 2: "Bachelor's", 3: "Master's", 4: "PhD+"}
DM02_LABELS = {1: "<1 yr", 2: "1–2 yr", 3: "3–5 yr", 4: "6–10 yr", 5: "10+ yr"}

print(f"Repo root : {REPO}")
print(f"CSV path  : {CSV_PATH}  (exists: {CSV_PATH.exists()})")
print(f"Key path  : {KEY_PATH}  (exists: {KEY_PATH.exists()})")
print(f"Batch dir : {BATCH_DIR} (exists: {BATCH_DIR.exists()})")

## 1. Load & Reshape Data

In [ ]:
# ── Cell 1: Load & Reshape ───────────────────────────────────────────────────

# ── Load Prolific demographics: keep only APPROVED participants ──────────────
prolific = pd.read_csv(PROLIFIC_PATH)
approved_ids = set(prolific.loc[prolific['Status'] == 'APPROVED', 'Participant id'])
print(f"Prolific: {len(prolific)} submissions, {len(approved_ids)} APPROVED")

# ── Load SoSci export ────────────────────────────────────────────────────────
raw = pd.read_csv(CSV_PATH, encoding='utf-16', sep='\t')
key = pd.read_csv(KEY_PATH)

# Apply cutoff: entries before this date are test runs, not real participants
CUTOFF_DATE = pd.to_datetime('2026-03-11 14:45:00')
raw['STARTED'] = pd.to_datetime(raw['STARTED'])
raw = raw[raw['STARTED'] >= CUTOFF_DATE].copy()

# Keep only finished responses from APPROVED Prolific participants
raw_fin = raw[
    (raw['FINISHED'] == 1) &
    (raw['REF'].isin(approved_ids))
].copy()
n_total, n_finished = len(raw), len(raw_fin)

# Parse TIME_RSI (German decimal comma → float)
raw_fin['TIME_RSI_num'] = (
    raw_fin['TIME_RSI'].astype(str).str.replace(',', '.').astype(float)
)

# ── Wide → Long ─────────────────────────────────────────────────────────────
rows = []
for _, r in raw_fin.iterrows():
    pid = r['REF']
    for pos in range(1, 21):
        idx_col = f'FD03x{pos:02d}'
        spec_col = f'FD01_{pos:02d}'
        gw_col   = f'FD02_{pos:02d}'
        sosci_idx = r[idx_col]
        if pd.isna(sosci_idx):
            continue
        sosci_idx = int(sosci_idx)
        item = key.iloc[sosci_idx - 1]  # 1-based → 0-based
        rows.append({
            'participant_id': pid,
            'loop_position':  pos,
            'sosci_index':    sosci_idx,
            'item_id':        item['item_id'],
            'text':           item['text'],
            'source':         item['source'],
            'is_gold':        bool(item['is_gold']),
            'gold_label':     item['gold_label'] if pd.notna(item['gold_label']) else None,
            'specificity':    r[spec_col],
            'greenwashing':   r[gw_col],
        })

long = pd.DataFrame(rows)

# ── Participant-level metadata ───────────────────────────────────────────────
participants = (
    raw_fin[['REF', 'STARTED', 'DM01', 'DM02', 'TIME_SUM', 'TIME_RSI_num', 'FD03_CP']]
    .rename(columns={'REF': 'participant_id'})
    .copy()
)
# Merge in Prolific demographics
participants = participants.merge(
    prolific[['Participant id', 'Time taken', 'Age', 'Sex']].rename(
        columns={'Participant id': 'participant_id', 'Time taken': 'prolific_time_secs'}
    ),
    on='participant_id', how='left'
)

# ── Summary ──────────────────────────────────────────────────────────────────
ratings_per_item = long.groupby('item_id').size()
print(f"After cutoff + APPROVED filter: {n_total} rows → {n_finished} finished")
print(f"Long-format ratings: {len(long)}  ({long['is_gold'].sum()} gold, {(~long['is_gold']).sum()} real)")
print(f"Unique items rated: {long['item_id'].nunique()} / {len(key)}")
print(f"Ratings per item:  min={ratings_per_item.min()}  median={ratings_per_item.median():.0f}  max={ratings_per_item.max()}")


## 2. Demographics & Timing

In [ ]:
# ── Cell 2: Demographics & Timing ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- Education (DM01) ---
edu_counts = participants["DM01"].dropna().astype(int).map(DM01_LABELS).value_counts()
edu_counts = edu_counts.reindex([v for v in DM01_LABELS.values() if v in edu_counts.index])
axes[0].barh(edu_counts.index, edu_counts.values, color=PAL[0])
axes[0].set_xlabel("Count")
axes[0].set_title("Education Level (DM01)")
for i, v in enumerate(edu_counts.values):
    axes[0].text(v + 0.3, i, str(v), va="center", fontweight="bold")

# --- Experience (DM02) ---
exp_counts = participants["DM02"].dropna().astype(int).map(DM02_LABELS).value_counts()
exp_counts = exp_counts.reindex([v for v in DM02_LABELS.values() if v in exp_counts.index])
axes[1].barh(exp_counts.index, exp_counts.values, color=PAL[1])
axes[1].set_xlabel("Count")
axes[1].set_title("Finance Experience (DM02)")
for i, v in enumerate(exp_counts.values):
    axes[1].text(v + 0.3, i, str(v), va="center", fontweight="bold")

# --- Completion time ---
time_min = participants["TIME_SUM"] / 60
axes[2].hist(time_min, bins=15, color=PAL[2], edgecolor="white", alpha=0.85)
axes[2].axvline(SPEED_THRESHOLD_SECS / 60, color="red", ls="--", lw=2,
                label=f"Speed cutoff ({SPEED_THRESHOLD_SECS}s)")
axes[2].axvline(time_min.median(), color="grey", ls=":", lw=1.5,
                label=f"Median ({time_min.median():.1f} min)")
axes[2].set_xlabel("Completion time (min)")
axes[2].set_ylabel("Count")
axes[2].set_title("Completion Time Distribution")
axes[2].legend(fontsize=9)

fig.tight_layout()
plt.savefig("/Users/maxim/Projects/gw-pilot-v1/outputs/figures/cell2_demographics_timing.pdf", bbox_inches="tight")
plt.show()

# ── Stats table ──────────────────────────────────────────────────────────────
print(f"\nCompletion time  —  mean: {time_min.mean():.1f} min  |  "
      f"median: {time_min.median():.1f} min  |  "
      f"min: {time_min.min():.1f} min  |  max: {time_min.max():.1f} min")
print(f"Participants below speed threshold ({SPEED_THRESHOLD_SECS}s): "
      f"{(participants['TIME_SUM'] < SPEED_THRESHOLD_SECS).sum()}")


## 3. Quality Filtering

Three-layer filter: (1) completion, (2) speed, (3) gold-item accuracy.  
All downstream analyses are run **with and without** flagged participants so the exclusion impact is transparent.

In [ ]:
# ── Cell 3: Quality Filtering ────────────────────────────────────────────────

# ── 3a. Gold-item accuracy per participant ───────────────────────────────────
gold_ratings = long[long["is_gold"]].copy()

def gold_correct(row):
    """Gold label 0 (vague) → specificity ≤ 2.5 is correct.
       Gold label 1 (specific) → specificity ≥ 3 is correct."""
    if row["gold_label"] == 0:
        return row["specificity"] <= 2.5
    else:
        return row["specificity"] >= 3

gold_ratings["correct"] = gold_ratings.apply(gold_correct, axis=1)
gold_acc = (
    gold_ratings.groupby("participant_id")["correct"]
    .agg(n_gold="count", n_correct="sum")
    .assign(gold_accuracy=lambda d: d["n_correct"] / d["n_gold"])
)

# ── 3b. Build participant QC table ───────────────────────────────────────────
qc = participants.merge(gold_acc, on="participant_id", how="left")

qc["flag_speed"] = qc["TIME_SUM"] < SPEED_THRESHOLD_SECS
qc["flag_gold"] = qc["gold_accuracy"] < GOLD_ACCURACY_THRESHOLD

qc["retained"] = ~(qc["flag_speed"] | qc["flag_gold"])
qc["exclusion_reason"] = ""
qc.loc[qc["flag_speed"] & qc["flag_gold"], "exclusion_reason"] = "speed + gold"
qc.loc[qc["flag_speed"] & ~qc["flag_gold"], "exclusion_reason"] = "speed"
qc.loc[~qc["flag_speed"] & qc["flag_gold"], "exclusion_reason"] = "gold"

# ── Display ──────────────────────────────────────────────────────────────────
display_cols = [
    "participant_id", "TIME_SUM", "TIME_RSI_num",
    "n_gold", "n_correct", "gold_accuracy",
    "flag_speed", "flag_gold", "retained", "exclusion_reason",
]
qc_display = qc[display_cols].sort_values("TIME_SUM").reset_index(drop=True)
qc_display["TIME_min"] = (qc_display["TIME_SUM"] / 60).round(1)

# Show flagged first, then retained
flagged = qc_display[~qc_display["retained"]]
print(f"=== FLAGGED PARTICIPANTS ({len(flagged)}) ===")
if len(flagged):
    display(flagged[["participant_id", "TIME_min", "gold_accuracy", "exclusion_reason"]])
else:
    print("  (none)")

n_retained = qc["retained"].sum()
print(f"\n--- Retention summary ---")
print(f"  Finished:           {n_finished}")
print(f"  Flagged (speed):    {qc['flag_speed'].sum()}")
print(f"  Flagged (gold):     {qc['flag_gold'].sum()}")
print(f"  Flagged (either):   {(~qc['retained']).sum()}")
print(f"  Retained:           {n_retained}")

# ── Build filtered long dataset ──────────────────────────────────────────────
retained_ids = set(qc.loc[qc["retained"], "participant_id"])
long_all = long.copy()  # keep unfiltered version for comparison
long_qc = long[long["participant_id"].isin(retained_ids)].copy()

print(f"\nRatings: {len(long_all)} total → {len(long_qc)} after QC")

## 4. Descriptive Statistics — Rating Distributions (after QC)

In [ ]:
# ── Cell 4: Descriptive Statistics ────────────────────────────────────────────
real_qc = long_qc[~long_qc["is_gold"]]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- 4a. Specificity distribution ---
spec_counts = real_qc["specificity"].value_counts().reindex(range(1, 6), fill_value=0)
axes[0].bar(spec_counts.index, spec_counts.values, color=PAL[0], edgecolor="white")
axes[0].set_xlabel("Specificity Rating")
axes[0].set_ylabel("Count")
axes[0].set_title("Specificity Ratings (real items, post-QC)")
axes[0].set_xticks(range(1, 6))
for x, v in zip(spec_counts.index, spec_counts.values):
    axes[0].text(x, v + 2, str(v), ha="center", fontweight="bold", fontsize=9)

# --- 4b. Greenwashing distribution ---
gw_counts = real_qc["greenwashing"].value_counts().reindex(range(1, 6), fill_value=0)
axes[1].bar(gw_counts.index, gw_counts.values, color=PAL[3], edgecolor="white")
axes[1].set_xlabel("Greenwashing Perception Rating")
axes[1].set_ylabel("Count")
axes[1].set_title("Greenwashing Perception (real items, post-QC)")
axes[1].set_xticks(range(1, 6))
for x, v in zip(gw_counts.index, gw_counts.values):
    axes[1].text(x, v + 2, str(v), ha="center", fontweight="bold", fontsize=9)

# --- 4c. Joint heatmap ---
joint = pd.crosstab(real_qc["specificity"], real_qc["greenwashing"])
joint = joint.reindex(index=range(1, 6), columns=range(1, 6), fill_value=0)
sns.heatmap(joint, annot=True, fmt="d", cmap="YlOrRd", ax=axes[2],
            cbar_kws={"label": "Count"})
axes[2].set_xlabel("Greenwashing Perception")
axes[2].set_ylabel("Specificity")
axes[2].set_title("Joint Distribution")
axes[2].invert_yaxis()

fig.tight_layout()
plt.savefig("/Users/maxim/Projects/gw-pilot-v1/outputs/figures/cell4_SPI_GW_bar.pdf", bbox_inches="tight")
plt.show()

# ── 4d. Per-item scatter: mean specificity vs mean greenwashing ──────────────
item_means = (
    real_qc.groupby("item_id")
    .agg(mean_spec=("specificity", "mean"), mean_gw=("greenwashing", "mean"),
         n=("specificity", "count"))
)

fig2, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(item_means["mean_spec"], item_means["mean_gw"],
                c=item_means["n"], cmap="viridis", s=40, alpha=0.7, edgecolors="grey", lw=0.5)
ax.set_xlabel("Mean Specificity Rating")
ax.set_ylabel("Mean Greenwashing Perception")
ax.set_title("Per-Item Means: Specificity vs Greenwashing (real items, post-QC)")
plt.colorbar(sc, ax=ax, label="N raters")

# Correlation annotation
r, p = stats.spearmanr(item_means["mean_spec"], item_means["mean_gw"])
ax.annotate(f"Spearman ρ = {r:.3f}  (p = {p:.2e})", xy=(0.05, 0.95),
            xycoords="axes fraction", fontsize=10, va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="grey", alpha=0.8))
fig2.tight_layout()

plt.savefig("/Users/maxim/Projects/gw-pilot-v1/outputs/figures/cell4_per_item_scatter.pdf", bbox_inches="tight")
plt.show()

print(f"Real items post-QC: {len(item_means)}  |  "
      f"Mean specificity: {item_means['mean_spec'].mean():.2f}  |  "
      f"Mean GW perception: {item_means['mean_gw'].mean():.2f}")

## 5. Inter-Rater Reliability — Krippendorff's Alpha

Alpha with **ordinal** distance function on the 5-point Likert scale.  
Interpretation: α ≥ 0.80 reliable, 0.67–0.80 tentative, < 0.67 unreliable (Krippendorff 2011).

In [ ]:
# ── Cell 5: Krippendorff's Alpha ─────────────────────────────────────────────

def build_reliability_matrix(df, dimension="specificity"):
    """Build rater × item matrix for krippendorff.alpha() (NaN = not seen)."""
    items  = sorted(df["item_id"].unique())
    raters = sorted(df["participant_id"].unique())
    item_idx  = {iid: i for i, iid in enumerate(items)}
    rater_idx = {rid: i for i, rid in enumerate(raters)}
    matrix = np.full((len(raters), len(items)), np.nan)
    for _, row in df.iterrows():
        matrix[rater_idx[row["participant_id"]], item_idx[row["item_id"]]] = row[dimension]
    return matrix

def compute_alpha(df, dimension):
    """Compute Krippendorff's Alpha (ordinal) for a given dimension."""
    matrix = build_reliability_matrix(df, dimension)
    return krippendorff.alpha(reliability_data=matrix, level_of_measurement="ordinal")

# ── Compute for both dimensions, with and without QC ─────────────────────────
results_alpha = {}
for label, df_subset in [("All participants", long_all), ("Post-QC", long_qc)]:
    real = df_subset[~df_subset["is_gold"]]  # real items only
    for dim in ["specificity", "greenwashing"]:
        alpha = compute_alpha(real, dim)
        results_alpha[(label, dim)] = alpha
        interp = "reliable" if alpha >= 0.80 else "tentative" if alpha >= 0.67 else "low"
        print(f"{label:20s} | {dim:14s} | α = {alpha:.4f}  ({interp})")

# ── Bar chart of alpha values ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
labels   = ["Specificity\n(all)", "GW Perception\n(all)",
             "Specificity\n(post-QC)", "GW Perception\n(post-QC)"]
alphas   = [
    results_alpha[("All participants", "specificity")],
    results_alpha[("All participants", "greenwashing")],
    results_alpha[("Post-QC", "specificity")],
    results_alpha[("Post-QC", "greenwashing")],
]
colors = [PAL[0], PAL[1], PAL[2], PAL[3]]
bars = ax.bar(labels, alphas, color=colors, edgecolor="white", width=0.55)
ax.axhline(0.67, color="grey", ls="--", lw=1.2, label="α = 0.67 (tentative threshold)")
ax.axhline(0.80, color="black", ls=":",  lw=1.2, label="α = 0.80 (reliable threshold)")
for bar, val in zip(bars, alphas):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}",
            ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Krippendorff's α")
ax.set_title("Inter-Rater Reliability — Krippendorff's α (ordinal)")
ax.set_ylim(0, 0.75)
ax.legend(fontsize=9)
fig.tight_layout()
plt.savefig("/Users/maxim/Projects/gw-pilot-v1/outputs/figures/cell5_inter-rater_reliability_bar.pdf", bbox_inches="tight")
plt.show()


## 6. Human–Pipeline Correlation (RQ1)

Match each survey item to the pipeline's **VUI** and **SPI** scores for the same firm-year.  
Two analyses: **(A)** real items only, **(B)** including gold items with `gold_label` as proxy.

> **Note on the Consistency Index (CI):** The GW composite formula in Part A (Eisinger) includes a third component — CI — measuring year-over-year emissions consistency using external Scope 1/2 emissions data (w\_CI = 0.20). CI operates on external financial/emissions data, not on text, so it is not applicable to sentence-level human validation. By design, when emissions data is unavailable the CI term is omitted and weights are redistributed automatically; it was active for DWS only (2 of 46 firm-year runs where Scope 1/2 data was available). This validation study therefore uses **VUI** and **SPI** as the two pipeline comparators — consistent with how the pipeline behaves for the 44 non-DWS firm-year runs.


In [ ]:
# ── Cell 6: Human–Pipeline Correlation ────────────────────────────────────────

# ── 6a. Load all pipeline batch results ──────────────────────────────────────
batch_rows = []
for firm_dir in sorted(BATCH_DIR.iterdir()):
    if not firm_dir.is_dir():
        continue
    for run_file in sorted(firm_dir.glob("run_*.json")):
        with open(run_file) as f:
            d = json.load(f)
        batch_rows.append({
            "firm": firm_dir.name,
            "year": d["year"],
            "gw": d["gw"]["gw"],
            "band": d["gw"]["band"],
            "vui_norm": d["vui"]["vui_norm"],
            "spi_rule": d["spi_rule"]["spi_rule"],
        })

batch_df = pd.DataFrame(batch_rows)

# ── 6b. Map survey key source → batch firm/year ─────────────────────────────
SOURCE_TO_FIRM = {
    "AMUNDI": "amundi", "BlackRock": "blackrock", "Blackstone": "blackstone",
    "CITIC": "citic", "Credit Suisse": "creditsuisse", "DWS": "dws",
    "IMPAX": "impax", "KKR": "kkr", "SCHRODERS": "schroders",
    "State Street": "state-street", "TRIODOS": "triodos", "UBS": "ubs",
}

def parse_source(source_str):
    """Parse 'AMUNDI 2023' → ('amundi', 2023)."""
    if pd.isna(source_str) or source_str == "Gold Standard":
        return None, None
    parts = source_str.rsplit(" ", 1)
    if len(parts) != 2:
        return None, None
    name, year_str = parts
    firm = SOURCE_TO_FIRM.get(name)
    if firm is None:
        return None, None
    return firm, int(year_str)

# ── 6c. Build item-level dataset with human means + pipeline scores ──────────
def build_correlation_data(df_long, include_gold=False):
    """Aggregate human ratings per item and merge with pipeline scores."""
    if include_gold:
        subset = df_long.copy()
    else:
        subset = df_long[~df_long["is_gold"]].copy()
    
    # Fill gold_label NaN with -1 so groupby doesn't drop real items
    subset["gold_label"] = subset["gold_label"].fillna(-1)
    
    item_agg = (
        subset.groupby(["item_id", "source", "is_gold", "gold_label", "text"])
        .agg(
            mean_spec=("specificity", "mean"),
            mean_gw=("greenwashing", "mean"),
            sd_spec=("specificity", "std"),
            sd_gw=("greenwashing", "std"),
            n_raters=("specificity", "count"),
        )
        .reset_index()
    )
    # Restore NaN for gold_label
    item_agg.loc[item_agg["gold_label"] == -1, "gold_label"] = np.nan
    
    # Parse source → firm + year
    firm_year = [parse_source(s) for s in item_agg["source"]]
    item_agg["firm"] = [fy[0] for fy in firm_year]
    item_agg["year"] = [fy[1] for fy in firm_year]
    item_agg["year"] = pd.to_numeric(item_agg["year"], errors="coerce")
    
    merged = item_agg.merge(
        batch_df[["firm", "year", "vui_norm", "spi_rule", "gw"]],
        on=["firm", "year"], how="left", suffixes=("_human", "_pipe"),
    )
    
    # For gold items in analysis B: use gold_label as proxy SPI (1 = specific, 0 = vague)
    if include_gold:
        gold_mask = merged["is_gold"]
        merged.loc[gold_mask, "spi_rule"] = merged.loc[gold_mask, "gold_label"]
        merged.loc[gold_mask, "vui_norm"] = 1.0 - merged.loc[gold_mask, "gold_label"]
    
    return merged

corr_A = build_correlation_data(long_qc, include_gold=False)
corr_B = build_correlation_data(long_qc, include_gold=True)

# Drop items with no pipeline match
corr_A_valid = corr_A.dropna(subset=["spi_rule", "vui_norm"])
corr_B_valid = corr_B.dropna(subset=["spi_rule", "vui_norm"])

print(f"Analysis A (real only):       {len(corr_A_valid)} items with pipeline match")
print(f"Analysis B (real + gold):     {len(corr_B_valid)} items with pipeline match")

In [ ]:
# ── Cell 6 (cont'd): Correlation statistics & scatter plots ──────────────────

def correlation_table(df, label):
    """Compute Spearman & Pearson for both dimension pairs."""
    rows = []
    for human_dim, pipe_dim, expected_sign in [
        ("mean_spec", "spi_rule", "+"),
        ("mean_gw", "vui_norm", "+"),
        ("mean_spec", "vui_norm", "−"),
        ("mean_gw", "spi_rule", "−"),
    ]:
        rho, p_rho = stats.spearmanr(df[human_dim], df[pipe_dim])
        r, p_r = stats.pearsonr(df[human_dim], df[pipe_dim])
        rows.append({
            "Analysis": label,
            "Human": human_dim, "Pipeline": pipe_dim,
            "Expected": expected_sign,
            "Spearman ρ": f"{rho:+.4f}", "p (Spearman)": f"{p_rho:.2e}",
            "Pearson r": f"{r:+.4f}", "p (Pearson)": f"{p_r:.2e}",
            "N": len(df),
        })
    return pd.DataFrame(rows)

corr_table_A = correlation_table(corr_A_valid, "A: Real only")
corr_table_B = correlation_table(corr_B_valid, "B: Real + Gold")

print("=== Correlation Results ===\n")
display(pd.concat([corr_table_A, corr_table_B], ignore_index=True))

# ── Scatter plots: main hypotheses ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

plot_configs = [
    (0, 0, "mean_spec", "spi_rule", "Mean Human Specificity", "Pipeline SPI (rule-based)",
     "H1: Specificity ↔ SPI  (real items)", corr_A_valid),
    (0, 1, "mean_gw", "vui_norm", "Mean Human GW Perception", "Pipeline VUI (normalised)",
     "H2: GW Perception ↔ VUI  (real items)", corr_A_valid),
    (1, 0, "mean_spec", "spi_rule", "Mean Human Specificity", "Pipeline SPI",
     "H1 + Gold items", corr_B_valid),
    (1, 1, "mean_gw", "vui_norm", "Mean Human GW Perception", "Pipeline VUI",
     "H2 + Gold items", corr_B_valid),
]

for row, col, hcol, pcol, xlabel, ylabel, title, data in plot_configs:
    ax = axes[row, col]
    
    # Colour gold items differently in analysis B
    if "Gold" in title:
        real_mask = ~data["is_gold"]
        ax.scatter(data.loc[real_mask, hcol], data.loc[real_mask, pcol],
                   alpha=0.5, s=30, color=PAL[0], label="Real", edgecolors="grey", lw=0.3)
        ax.scatter(data.loc[~real_mask, hcol], data.loc[~real_mask, pcol],
                   alpha=0.8, s=50, color=PAL[3], marker="D", label="Gold", edgecolors="grey", lw=0.3)
        ax.legend(fontsize=8)
    else:
        ax.scatter(data[hcol], data[pcol], alpha=0.5, s=30, color=PAL[0], edgecolors="grey", lw=0.3)
    
    # OLS trend line
    mask = data[[hcol, pcol]].dropna().index
    x, y = data.loc[mask, hcol], data.loc[mask, pcol]
    z = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, np.polyval(z, xline), color="red", ls="--", lw=1.5, alpha=0.7)
    
    rho, p = stats.spearmanr(x, y)
    ax.annotate(f"ρ = {rho:+.3f}  (p = {p:.2e})", xy=(0.05, 0.95),
                xycoords="axes fraction", fontsize=9, va="top",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="grey", alpha=0.8))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

fig.suptitle("Human Ratings vs Pipeline Scores", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig("/Users/maxim/Projects/gw-pilot-v1/outputs/figures/cell6_correlation_human-pipeline_scatter.pdf", bbox_inches="tight")
plt.show()

## 7. Item-Level Divergence Analysis

Where do human ratings and pipeline scores disagree most? Examining these items reveals what the pipeline misses or over-counts.

In [ ]:
# ── Cell 7: Item-Level Divergence ─────────────────────────────────────────────

# Normalise both scales to [0, 1] for comparability
# Human specificity: (mean_spec - 1) / 4  → 0 = "not at all", 1 = "very"
# Pipeline SPI: already 0–1 (fraction of specific sentences)

div = corr_A_valid.copy()

# Ensure text column is present (join from key if dropped by groupby)
if "text" not in div.columns:
    div = div.merge(key[["item_id", "text"]].drop_duplicates(), on="item_id", how="left")

div["human_spec_norm"] = (div["mean_spec"] - 1) / 4
div["human_gw_norm"] = (div["mean_gw"] - 1) / 4

# Divergence = human - pipeline (positive = human rates higher than pipeline)
div["div_spec"] = div["human_spec_norm"] - div["spi_rule"]
div["div_gw"] = div["human_gw_norm"] - div["vui_norm"]

# ── Top divergent items: Specificity ─────────────────────────────────────────
print("=== Top 10 divergent items: Human Specificity vs Pipeline SPI ===")
print("(Positive = humans rate MORE specific than pipeline)\n")
top_spec = div.nlargest(5, "div_spec")[["item_id", "text", "mean_spec", "spi_rule", "div_spec", "source"]]
bot_spec = div.nsmallest(5, "div_spec")[["item_id", "text", "mean_spec", "spi_rule", "div_spec", "source"]]
divergent_spec = pd.concat([top_spec, bot_spec])
for _, row in divergent_spec.iterrows():
    direction = "HUMAN > PIPELINE" if row["div_spec"] > 0 else "PIPELINE > HUMAN"
    print(f"  [{row['item_id']}] {direction}  (human={row['mean_spec']:.1f}, SPI={row['spi_rule']:.3f})")
    print(f"    \"{row['text'][:120]}...\"" if len(str(row['text'])) > 120 else f"    \"{row['text']}\"")
    print()

# ── Top divergent items: Greenwashing ────────────────────────────────────────
print("\n=== Top 10 divergent items: Human GW Perception vs Pipeline VUI ===")
print("(Positive = humans perceive MORE greenwashing than pipeline)\n")
top_gw = div.nlargest(5, "div_gw")[["item_id", "text", "mean_gw", "vui_norm", "div_gw", "source"]]
bot_gw = div.nsmallest(5, "div_gw")[["item_id", "text", "mean_gw", "vui_norm", "div_gw", "source"]]
divergent_gw = pd.concat([top_gw, bot_gw])
for _, row in divergent_gw.iterrows():
    direction = "HUMAN > PIPELINE" if row["div_gw"] > 0 else "PIPELINE > HUMAN"
    print(f"  [{row['item_id']}] {direction}  (human={row['mean_gw']:.1f}, VUI={row['vui_norm']:.3f})")
    print(f"    \"{row['text'][:120]}...\"" if len(str(row['text'])) > 120 else f"    \"{row['text']}\"")
    print()

# ── Divergence distribution plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(div["div_spec"], bins=25, color=PAL[0], edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="black", ls="-", lw=1)
axes[0].set_xlabel("Divergence (human − pipeline)")
axes[0].set_ylabel("Number of items")
axes[0].set_title("Specificity: Human vs SPI")

axes[1].hist(div["div_gw"], bins=25, color=PAL[3], edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="black", ls="-", lw=1)
axes[1].set_xlabel("Divergence (human − pipeline)")
axes[1].set_ylabel("Number of items")
axes[1].set_title("GW Perception: Human vs VUI")

fig.suptitle("Distribution of Human–Pipeline Divergence (normalised to [0,1])", fontweight="bold")
fig.tight_layout()
plt.savefig("/Users/maxim/Projects/gw-pilot-v1/outputs/figures/cell7_human-spi_pipeline-spi_bar.pdf", bbox_inches="tight")
plt.show()


## 8. SFDR Sub-sample Comparison (RQ2)

Compare pipeline scores for the 5 EU firms subject to SFDR. Hypothesis: sustainability-focused firms (Triodos, Impax) should show higher SPI than mixed managers.

In [ ]:
# ── Cell 8: SFDR Sub-sample Comparison ────────────────────────────────────────

EU5 = ["amundi", "dws", "schroders", "triodos", "impax"]
SFDR_PROFILE = {
    "amundi": "Mixed", "dws": "Mixed", "schroders": "Mixed",
    "triodos": "Sustainability-focused", "impax": "Sustainability-focused",
}

eu_batch = batch_df[batch_df["firm"].isin(EU5)].copy()
eu_batch["sfdr_profile"] = eu_batch["firm"].map(SFDR_PROFILE)

# ── Firm-level means ─────────────────────────────────────────────────────────
eu_means = (
    eu_batch.groupby(["firm", "sfdr_profile"])
    .agg(mean_gw=("gw", "mean"), mean_vui=("vui_norm", "mean"),
         mean_spi=("spi_rule", "mean"), n_years=("year", "count"))
    .reset_index()
    .sort_values("mean_gw")
)
print("=== EU5 SFDR Sub-sample: Pipeline Scores (mean across years) ===\n")
display(eu_means[["firm", "sfdr_profile", "mean_gw", "mean_vui", "mean_spi", "n_years"]])

# ── Grouped bar chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
firms_ordered = eu_means.sort_values("mean_gw")["firm"].tolist()
profile_colors = {"Sustainability-focused": PAL[2], "Mixed": PAL[0]}

for i, (metric, title) in enumerate([
    ("mean_gw", "Composite GW Score"),
    ("mean_vui", "VUI (Vagueness)"),
    ("mean_spi", "SPI (Specificity)"),
]):
    ax = axes[i]
    bars = ax.barh(
        [f.title() for f in firms_ordered],
        eu_means.set_index("firm").loc[firms_ordered, metric],
        color=[profile_colors[SFDR_PROFILE[f]] for f in firms_ordered],
        edgecolor="white",
    )
    ax.set_xlabel(metric.replace("mean_", "").upper())
    ax.set_title(title)
    for bar, val in zip(bars, eu_means.set_index("firm").loc[firms_ordered, metric]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=profile_colors[p], label=p) for p in profile_colors]
fig.legend(handles=legend_handles, loc="upper right", fontsize=9, framealpha=0.9)
fig.suptitle("SFDR Sub-sample: EU5 Pipeline Scores", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.88, 0.95])
plt.savefig("/Users/maxim/Projects/gw-pilot-v1/outputs/figures/cell8_SFDR-sample_bar.pdf", bbox_inches="tight")
plt.show()

# ── Mann-Whitney U: sustainability-focused vs mixed ──────────────────────────
focused = eu_batch[eu_batch["sfdr_profile"] == "Sustainability-focused"]
mixed = eu_batch[eu_batch["sfdr_profile"] == "Mixed"]

print("\n=== Mann-Whitney U: Sustainability-focused vs Mixed ===")
for metric, label in [("gw", "GW"), ("vui_norm", "VUI"), ("spi_rule", "SPI")]:
    u, p = stats.mannwhitneyu(focused[metric], mixed[metric], alternative="two-sided")
    print(f"  {label:4s}: U = {u:.1f}, p = {p:.4f}  "
          f"(focused median = {focused[metric].median():.3f}, "
          f"mixed median = {mixed[metric].median():.3f})")

# ── Per-year trend: EU5 pipeline scores over time ────────────────────────────
eu_yearly = eu_batch.sort_values(["firm", "year"])

# Shades per firm within each SFDR profile group
FIRM_COLORS = {
    "amundi": "#1a5276", "dws": "#2e86c1", "schroders": "#85c1e9",   # Mixed — blue shades
    "triodos": "#145a32", "impax": "#27ae60",                         # Sust-focused — green shades
}
FIRM_MARKERS = {"amundi": "s", "dws": "D", "schroders": "^", "triodos": "o", "impax": "P"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
for i, (metric, title, ylabel) in enumerate([
    ("gw", "Composite GW Score", "GW"),
    ("vui_norm", "VUI (Vagueness)", "VUI"),
    ("spi_rule", "SPI (Specificity)", "SPI"),
]):
    ax = axes[i]
    for firm in EU5:
        fd = eu_yearly[eu_yearly["firm"] == firm]
        ls = "--" if SFDR_PROFILE[firm] == "Mixed" else "-"
        ax.plot(fd["year"], fd[metric], ls=ls, marker=FIRM_MARKERS[firm],
                color=FIRM_COLORS[firm], label=firm.title(), lw=2, markersize=7)
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(sorted(eu_batch["year"].unique()))

axes[0].legend(fontsize=8, loc="best")
fig.suptitle("EU5 SFDR Sub-sample: Pipeline Scores by Year\n(solid/green = sustainability-focused, dashed/blue = mixed)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig("/Users/maxim/Projects/gw-pilot-v1/outputs/figures/cell8_SFDR-sample-by-year_bar.pdf", bbox_inches="tight")
plt.show()


## 9. Counterbalance & Order Effects

Check whether the urn-draw condition (FD03_CP) or item position (1–20) systematically affect ratings.

In [ ]:
# ── Cell 9: Counterbalance & Order Effects ────────────────────────────────────

# Merge counterbalance condition into long data
long_qc_ext = long_qc.merge(
    participants[["participant_id", "FD03_CP"]], on="participant_id", how="left"
)
real_ext = long_qc_ext[~long_qc_ext["is_gold"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 9a. Counterbalance condition effect ──────────────────────────────────────
cp_groups = [g["specificity"].values for _, g in real_ext.groupby("FD03_CP")]
H_cp, p_cp = stats.kruskal(*cp_groups)

sns.boxplot(data=real_ext, x="FD03_CP", y="specificity", ax=axes[0],
            palette="Set2", width=0.5)
axes[0].set_xlabel("Counterbalance Condition (FD03_CP)")
axes[0].set_ylabel("Specificity Rating")
axes[0].set_title(f"Specificity by Condition  (Kruskal-Wallis H={H_cp:.2f}, p={p_cp:.4f})")

# ── 9b. Position / order effect ──────────────────────────────────────────────
pos_means = real_ext.groupby("loop_position")["specificity"].mean()
axes[1].plot(pos_means.index, pos_means.values, marker="o", color=PAL[0], lw=2)
axes[1].fill_between(pos_means.index,
                     real_ext.groupby("loop_position")["specificity"].quantile(0.25),
                     real_ext.groupby("loop_position")["specificity"].quantile(0.75),
                     alpha=0.2, color=PAL[0])
axes[1].set_xlabel("Item Position (1 = first shown)")
axes[1].set_ylabel("Mean Specificity Rating")

r_pos, p_pos = stats.spearmanr(real_ext["loop_position"], real_ext["specificity"])
axes[1].set_title(f"Position Effect  (Spearman ρ={r_pos:+.3f}, p={p_pos:.4f})")
axes[1].set_xticks(range(1, 21))

fig.suptitle("Survey Design Checks", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

# ── Same for greenwashing dimension ──────────────────────────────────────────
cp_groups_gw = [g["greenwashing"].values for _, g in real_ext.groupby("FD03_CP")]
H_cp_gw, p_cp_gw = stats.kruskal(*cp_groups_gw)
r_pos_gw, p_pos_gw = stats.spearmanr(real_ext["loop_position"], real_ext["greenwashing"])

print(f"Greenwashing — Condition effect:  H = {H_cp_gw:.2f}, p = {p_cp_gw:.4f}")
print(f"Greenwashing — Position effect:   ρ = {r_pos_gw:+.3f}, p = {p_pos_gw:.4f}")

## 10. Summary Table for Thesis

All key statistics in one table, ready for LaTeX or thesis copy-paste.

In [ ]:
# ── Cell 10: Summary Table ────────────────────────────────────────────────────

# results_alpha stores floats (point estimate only, no CI)
def _alpha_str(label, dim):
    a = results_alpha[(label, dim)]
    interp = "reliable" if a >= 0.80 else "tentative" if a >= 0.67 else "low"
    return f"{a:.4f} ({interp})"

# Spearman from Analysis A (real items, post-QC)
rho_spec_spi, p_spec_spi = stats.spearmanr(corr_A_valid["mean_spec"], corr_A_valid["spi_rule"])
rho_gw_vui,  p_gw_vui  = stats.spearmanr(corr_A_valid["mean_gw"],   corr_A_valid["vui_norm"])

summary = pd.DataFrame([
    {"Statistic": "N participants (finished)",          "Value": str(n_finished)},
    {"Statistic": "N retained (post-QC)",              "Value": str(n_retained)},
    {"Statistic": "N excluded (speed)",                "Value": str(int(qc["flag_speed"].sum()))},
    {"Statistic": "N excluded (gold accuracy)",        "Value": str(int(qc["flag_gold"].sum()))},
    {"Statistic": "N items rated (real)",              "Value": str(long_qc[~long_qc["is_gold"]]["item_id"].nunique())},
    {"Statistic": "N items rated (gold)",              "Value": str(long_qc[long_qc["is_gold"]]["item_id"].nunique())},
    {"Statistic": "Median ratings per item",           "Value": f"{long_qc.groupby('item_id').size().median():.0f}"},
    {"Statistic": "Speed threshold",                   "Value": f"{SPEED_THRESHOLD_SECS}s"},
    {"Statistic": "Gold accuracy threshold",           "Value": f"{GOLD_ACCURACY_THRESHOLD:.0%}"},
    {"Statistic": "",                                  "Value": ""},
    {"Statistic": "Krippendorff α — Specificity (all)",      "Value": _alpha_str("All participants", "specificity")},
    {"Statistic": "Krippendorff α — GW Perception (all)",    "Value": _alpha_str("All participants", "greenwashing")},
    {"Statistic": "Krippendorff α — Specificity (post-QC)",  "Value": _alpha_str("Post-QC", "specificity")},
    {"Statistic": "Krippendorff α — GW Perception (post-QC)","Value": _alpha_str("Post-QC", "greenwashing")},
    {"Statistic": "",                                  "Value": ""},
    {"Statistic": "Spearman ρ: Human Specificity ↔ Pipeline SPI", "Value": f"{rho_spec_spi:+.4f} (p = {p_spec_spi:.2e})"},
    {"Statistic": "Spearman ρ: Human GW Perception ↔ Pipeline VUI", "Value": f"{rho_gw_vui:+.4f} (p = {p_gw_vui:.2e})"},
    {"Statistic": "",                                  "Value": ""},
    {"Statistic": "Condition effect (Kruskal-Wallis p)", "Value": f"Spec: {p_cp:.4f}  |  GW: {p_cp_gw:.4f}"},
    {"Statistic": "Position effect (Spearman p)",       "Value": f"Spec: {p_pos:.4f}  |  GW: {p_pos_gw:.4f}"},
    {"Statistic": "",                                  "Value": ""},
    {"Statistic": "Note: Consistency Index (CI)",       "Value": "Not applicable for sentence-level validation — CI uses external Scope 1/2 emissions data, active for DWS only (2/46 runs) by design"},
])

display(summary.style.hide(axis="index").set_properties(**{"text-align": "left"}))

print("\n--- LaTeX-friendly (copy-paste) ---\n")
for _, row in summary.iterrows():
    if row["Statistic"]:
        print(f"  {row['Statistic']} & {row['Value']} \\\\")
